# Notebook 02 — LINCS L1000 Connectivity Scoring

**Goal:** For each MoTrPAC tissue, query the LINCS L1000 database to find
drug/compound signatures that are most similar to the exercise gene signature.
This is Phase 2 of the pipeline — the minimum deliverable.

**Method:** POST up/down gene sets to SigCom LINCS / L2S2 API, aggregate
per-experiment enrichment scores to compound-level ranks, apply cell-line
tissue weighting, and generate a meta-rank across tissues.

**Outputs:**
- `data/processed/lincs_raw_{tissue}.parquet` — cached API responses
- `data/processed/lincs_ranked_{tissue}.parquet` — compound-level ranked list
- `results/ranked_compounds.csv` — merged meta-ranking across tissues

**Sanity gate:** AICAR, metformin, GW501516, resveratrol, or bezafibrate
should appear in the top 200 hits for at least one tissue.

In [ ]:
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent))
from src import signatures as sig
from src import connectivity as conn

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

# ── Configuration ─────────────────────────────────────────────────────────────
# Set USE_LOCAL_GCTX = True only if you have downloaded the ~50GB GCTX file.
USE_LOCAL_GCTX  = False
API_CHOICE      = 'l2s2'   # 'l2s2' | 'sigcom'
FORCE_REQUERY   = False    # True = ignore cache, re-run API calls

TISSUES      = list(sig.TISSUES.keys())
TIMEPOINT    = '8w'
TOP_N_AGG    = 5000        # max signatures to pull from API
TOP_N_REPORT = 500         # compounds in final ranked output

PROCESSED_DIR  = Path('../data/processed')
RESULTS_DIR    = Path('../results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'API: {API_CHOICE}  |  FORCE_REQUERY: {FORCE_REQUERY}')
print(f'Tissues: {TISSUES}')

In [ ]:
# ── Load processed gene sets from notebook 01 ─────────────────────────────────

gene_sets = {}
for tissue in TISSUES:
    up_path   = PROCESSED_DIR / f'motrpac_{tissue}_{TIMEPOINT}_up.txt'
    down_path = PROCESSED_DIR / f'motrpac_{tissue}_{TIMEPOINT}_down.txt'

    if not up_path.exists() or not down_path.exists():
        raise FileNotFoundError(
            f'Gene sets not found for {tissue}. Run notebook 01 first.'
        )

    gene_sets[tissue] = {
        'up':   sig.load_gene_set(up_path),
        'down': sig.load_gene_set(down_path),
    }
    print(f'{tissue}: {len(gene_sets[tissue]["up"])} up, '
          f'{len(gene_sets[tissue]["down"])} down genes')

print('Gene sets loaded.')

In [ ]:
# ── Query LINCS for each tissue ───────────────────────────────────────────────
# Results are cached to parquet after the first run.
# DO NOT re-run without caching — API rate limits will bite you.

raw_results = {}

for tissue in TISSUES:
    print(f'\n=== {tissue} ===')
    up_genes   = gene_sets[tissue]['up']
    down_genes = gene_sets[tissue]['down']
    cache_path = PROCESSED_DIR / f'lincs_raw_{tissue}.parquet'

    if USE_LOCAL_GCTX:
        from src.connectivity import compute_connectivity_local
        RAW_DIR = Path('../data/raw')
        df = compute_connectivity_local(
            up_genes, down_genes,
            gctx_path=RAW_DIR / 'GSE92742_Broad_LINCS_Level5_COMPZ.MODZ_n473647x12328.gctx',
            gene_info_path=RAW_DIR / 'GSE92742_Broad_LINCS_gene_info.txt.gz',
            sig_info_path=RAW_DIR / 'GSE92742_Broad_LINCS_sig_info.txt.gz',
        )
    else:
        df = conn.cache_query(
            up_genes, down_genes,
            cache_path=cache_path,
            force=FORCE_REQUERY,
            api=API_CHOICE,
        )

    raw_results[tissue] = df
    print(f'  Loaded {len(df):,} raw signatures')
    if not df.empty:
        print(df.head(3).to_string())

print('\nAll tissues queried.')

In [ ]:
# ── Aggregate per-experiment → per-compound scores ────────────────────────────

ranked = {}   # {tissue: compound-level DataFrame}

for tissue in TISSUES:
    df = raw_results[tissue]
    if df.empty:
        print(f'No data for {tissue}, skipping.')
        ranked[tissue] = pd.DataFrame()
        continue

    # Two aggregation strategies: simple mean and cell-line weighted
    agg_simple   = conn.aggregate_by_compound(df)
    agg_weighted = conn.cell_line_weighted_score(df, tissue)

    # Merge to get both scores
    merged = agg_simple.merge(
        agg_weighted[['pert_iname', 'weighted_median_score']],
        on='pert_iname', how='left'
    )
    merged['tissue'] = tissue
    ranked[tissue] = merged.head(TOP_N_REPORT)

    # Save ranked list
    merged.to_parquet(PROCESSED_DIR / f'lincs_ranked_{tissue}.parquet', index=False)

    print(f'\n{tissue}: top 10 compounds')
    score_col = 'median_score' if 'median_score' in merged.columns else merged.columns[1]
    cols_show = ['pert_iname', score_col, 'n_signatures', 'moa', 'target', 'cell_lines']
    cols_show = [c for c in cols_show if c in merged.columns]
    print(merged.head(10)[cols_show].to_string(index=False))

    # Sanity check
    sanity = conn.sanity_check_ranking(merged, tissue)
    print(f'  Known mimetics in top-200: {sanity["known_mimetics_in_top200"]}')

In [ ]:
# ── Meta-ranking across tissues ───────────────────────────────────────────────
# Assign rank per tissue, then average ranks (Borda-style) across tissues.
# Compounds appearing in multiple tissues get boosted.

all_dfs = []
for tissue, df in ranked.items():
    if df.empty:
        continue
    score_col = 'median_score' if 'median_score' in df.columns else df.columns[1]
    tmp = df[['pert_iname', score_col, 'moa', 'target']].copy()
    tmp['tissue'] = tissue
    tmp['score'] = tmp[score_col]
    tmp['rank_in_tissue'] = tmp['score'].rank(ascending=False, method='min')
    all_dfs.append(tmp)

if not all_dfs:
    print('No results to meta-rank.')
else:
    long_df = pd.concat(all_dfs, ignore_index=True)

    meta = (
        long_df.groupby('pert_iname')
        .agg(
            n_tissues      = ('tissue', 'count'),
            mean_rank      = ('rank_in_tissue', 'mean'),
            mean_score     = ('score', 'mean'),
            max_score      = ('score', 'max'),
            tissues_ranked = ('tissue', lambda x: '|'.join(sorted(x))),
            moa            = ('moa', lambda x: x.dropna().mode().iloc[0]
                              if x.dropna().size else ''),
            target         = ('target', lambda x: x.dropna().mode().iloc[0]
                              if x.dropna().size else ''),
        )
        .reset_index()
        .sort_values(['n_tissues', 'mean_rank'], ascending=[False, True])
        .reset_index(drop=True)
    )
    meta['meta_rank'] = range(1, len(meta) + 1)

    # Save final ranked compound list
    meta.to_csv(RESULTS_DIR / 'ranked_compounds.csv', index=False)
    meta.to_parquet(PROCESSED_DIR / 'ranked_compounds_meta.parquet', index=False)

    print('Top 20 meta-ranked compounds:')
    print(meta.head(20)[['meta_rank', 'pert_iname', 'mean_score', 'n_tissues',
                           'tissues_ranked', 'moa']].to_string(index=False))

In [ ]:
# ── Score distribution plots ──────────────────────────────────────────────────

fig, axes = plt.subplots(1, len([t for t in TISSUES if not ranked.get(t, pd.DataFrame()).empty]),
                          figsize=(5 * len(TISSUES), 4))
if not isinstance(axes, np.ndarray):
    axes = [axes]

ax_iter = iter(axes)
for tissue in TISSUES:
    df = ranked.get(tissue, pd.DataFrame())
    if df.empty:
        continue
    ax = next(ax_iter)
    score_col = 'median_score' if 'median_score' in df.columns else df.columns[1]

    scores = df[score_col].dropna()
    ax.hist(scores, bins=50, color='steelblue', edgecolor='white', alpha=0.8)

    # Highlight known mimetics
    known = df[df['pert_iname'].str.lower().isin(
        {m.lower() for m in conn.KNOWN_MIMETICS}
    )]
    for _, row in known.iterrows():
        ax.axvline(row[score_col], color='firebrick', alpha=0.7, lw=1.2)
        ax.text(row[score_col], ax.get_ylim()[1] * 0.9, row['pert_iname'],
                rotation=90, fontsize=7, color='firebrick', ha='right')

    ax.set_title(f'{sig.TISSUES[tissue]["display"]}')
    ax.set_xlabel('Connectivity score (median z-score)')
    ax.set_ylabel('Count')

plt.suptitle('LINCS connectivity score distributions', fontsize=12)
plt.tight_layout()
plt.savefig('../results/figures/02_score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved score distribution plots.')

In [ ]:
# ── Tissue overlap analysis ───────────────────────────────────────────────────
# How many top-100 compounds are shared across all 3 tissues?

from matplotlib_venn import venn3

top100_sets = {}
for tissue in TISSUES:
    df = ranked.get(tissue, pd.DataFrame())
    if df.empty:
        top100_sets[tissue] = set()
    else:
        score_col = 'median_score' if 'median_score' in df.columns else df.columns[1]
        top100_sets[tissue] = set(df.head(100)['pert_iname'])

if len(TISSUES) == 3:
    fig, ax = plt.subplots(figsize=(6, 6))
    labels = [sig.TISSUES[t]['display'].split('(')[0].strip() for t in TISSUES]
    try:
        venn3([top100_sets[t] for t in TISSUES], set_labels=labels, ax=ax)
        ax.set_title('Top-100 compound overlap across tissues')
        plt.savefig('../results/figures/02_tissue_overlap_venn.png', dpi=150, bbox_inches='tight')
        plt.show()
    except ImportError:
        # matplotlib_venn not installed — print overlap counts
        sets = [top100_sets[t] for t in TISSUES]
        print('Venn diagram (install matplotlib-venn for plot):')
        print(f'  Only {TISSUES[0]}: {len(sets[0] - sets[1] - sets[2])}')
        print(f'  Only {TISSUES[1]}: {len(sets[1] - sets[0] - sets[2])}')
        print(f'  Only {TISSUES[2]}: {len(sets[2] - sets[0] - sets[1])}')
        print(f'  All 3 tissues: {len(sets[0] & sets[1] & sets[2])}')
        shared = sets[0] & sets[1] & sets[2]
        print(f'\nShared top-100 compounds: {sorted(shared)[:20]}')

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────

print('=== Notebook 02 complete ===')
print()
for tissue in TISSUES:
    df = ranked.get(tissue, pd.DataFrame())
    print(f'{tissue}: {len(df)} compounds ranked')
print()
if not all_dfs:
    print('No meta-ranking produced (empty results).')
else:
    print(f'Meta-ranking: {len(meta)} unique compounds across all tissues')
    known_in_top100 = meta.head(100)['pert_iname'].str.lower()
    found = [m for m in conn.KNOWN_MIMETICS if m.lower() in set(known_in_top100)]
    print(f'Known mimetics in meta-top-100: {found}')
print()
print('Files written:')
for f in sorted(PROCESSED_DIR.glob('lincs_*.parquet')):
    print(f'  {f}')
print(f'  {RESULTS_DIR / "ranked_compounds.csv"}')
print()
print('→ Next: run notebook 03_gsfm_embedding.ipynb')